# CIS 5200: Machine Learning
## Homework 4 (Programming)

In [171]:
import os
import sys

NOTEBOOK = (os.getenv('IS_AUTOGRADER') is None)
if NOTEBOOK:
    print("[INFO, OK] Google Colab.")
else:
    print("[INFO, OK] Autograder.")
    sys.exit()

[INFO, OK] Google Colab.


### Penngrader setup

In [172]:
# %%capture
!pip install penngrader-client

In [173]:
%%writefile config.yaml
grader_api_url: 'https://23whrwph9h.execute-api.us-east-1.amazonaws.com/default/Grader23'
grader_api_key: 'flfkE736fA6Z8GxMDJe2q8Kfk8UDqjsG3GVqOFOa'

Overwriting config.yaml


In [174]:
# packages for homework
import torch
import torch.nn.functional as F
import torch.nn as nn

from sklearn import datasets
from sklearn.model_selection import train_test_split

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

## 1. Decision Trees and Bagging (10pts)

In this problem, we'll implement a simplified version of random forests. We'll be using the `iris` dataset, which has 4 features that are discretized to $0.1$ steps between $0$ and $8$ (i.e. the set of all possible features is $\{0.1, 0.2, \dots, 7.8, 7.9\}$. Thus, all thresholds that we'll need to consider are $\{0.15, 0.25, \dots, 7.75, 7.85\}$.

Your task in this first part is to finish the implementation of decision trees. We've provided a template for some of the decision tree code, following which you'll finish the bagging algorithm to get a random forest.

1. Entropy (2pts): calculate the entropy of a given vector of labels in the `entropy` function. Note that the generalization of entropy to 3 classes is $H = -\sum_{i=1}^3 p_i\log(p_i)$ where $p_i$ is the proportion of examples with label $i$.
2. Information gain (1pt): finish the `find_split` function by calculating the criteria that is used to determine the best split, often called information gain.
3. Build the tree (2pts): finish the `expand_node` function by completing the recursive call for building the decision tree.
4. Predict with tree (2pts): implement the `predict_one` function, which makes a prediction for a single example.

Tips:
+ If you have NaNs, you may be dividing by zero.


In [175]:
def entropy(y):
    values, counts = y.unique(return_counts=True)
    probabilities = counts.float() / counts.sum()
    probabilities = probabilities[probabilities > 0]
    return -torch.sum(probabilities * torch.log2(probabilities)).item()

def find_split(node, X, y, k=4):
    H,m = entropy(y), y.size(0)
    best_IG, best_split = -999, None
    features = torch.randint(4,(k,))

    for feature_idx in features:
        for threshold in torch.arange(0.15,7.9,0.1):
            idx = X[:,feature_idx] > threshold
            if idx.sum() == 0 or idx.sum() == idx.size(0):
                continue

            m_left = idx.sum()
            m_right = (~idx).sum()

            H_left = entropy(y[idx])
            H_right = entropy(y[~idx])

            ## ANSWER TODO
            IG = H - ((m_left / m) * H_left + (m_right / m) * H_right)
            ## END ANSWER

            if IG > best_IG or best_split == None:
                best_IG, best_split = IG, (feature_idx, threshold)
    return best_split

def expand_node(node, X, y, max_depth=0, k=4):
    H = entropy(y)
    if H == 0 or max_depth == 0:
        return

    best_split = find_split(node, X, y, k=k)

    if best_split == None:
        return

    idx = X[:,best_split[0]] > best_split[1]
    X_left, y_left = X[idx], y[idx]
    X_right, y_right = X[~idx], y[~idx]

    del node['label']
    node['split'] = best_split
    node['left'] = { 'label': y_left.mode().values }
    node['right'] = { 'label': y_right.mode().values }

    ## ANSWER TODO
    expand_node(node['left'], X_left, y_left, max_depth - 1)
    expand_node(node['right'], X_right, y_right, max_depth - 1)
    ## END ANSWER

    return

def fit_decision_tree(X,y, k=4):
    root = { 'label': y.mode().values }
    expand_node(root, X, y, max_depth=10, k=k)
    return root

def predict_one(node, x):
    if 'label' in node:
        return node['label']
    feature_index, threshold = node['split']
    if x[feature_index] >= threshold:
        return predict_one(node['left'], x)
    else:
        return predict_one(node['right'], x)



def predict(node, X):
    return torch.stack([predict_one(node, x) for x in X])

Test your code on the `iris` dataset. Your decision tree should fit to 100\% training accuracy and generalize to 88\% test accuracy.

In [176]:
iris = datasets.load_iris()
data=train_test_split(iris.data,iris.target,test_size=0.5,random_state=123)

X,X_te,y,y_te = [torch.from_numpy(A) for A in data]
X,X_te,y,y_te = X.float(), X_te.float(), y.long(), y_te.float()

DT = fit_decision_tree(X,y,k=4)
print('Train accuracy: ', (predict(DT, X) == y).float().mean().item())
print('Test accuracy: ', (predict(DT, X_te) == y_te).float().mean().item())

Train accuracy:  1.0
Test accuracy:  0.9333333373069763


### Bagging Decision Trees for Random forests (3pts)

Note that our `find_split` implementation can use a random subset of the features when searching for the right split via the argument $k$. For the vanilla decision tree, we defaulted to $k=4$. Since there were 4 features, this meant that the decision tree could always considered all 4 features. By reducing the value of $k$ to $\sqrt(k)=2$, we can introduce variance into the decision trees for the bagging algorithm.

You'll now implement the bagging algorithm. Note that if you use the `clf` and `predict` functions given as keyword arguments, you can pass this section in the autograder without needing a correct implementation for decision trees from the previous section.

1. Bootstrap (1pt): Implement `bootstrap` to draw a random bootstrap dataset from the given dataset.
2. Fitting a random forest (1pt): Implement `random_forest_fit` to train a random forest that fits the data.
3. Predicting with a random forest (1pt): Implement `predict_forest_fit` to make predictions given a random forest.

Tip:
+ If you're not sure whether your bootstrap is working or not, remember that on average, there will be $1-1/e\approx 0.632$ unique samples in a bootstrapped dataset.

In [177]:
def bootstrap(X,y):
    m = X.size(0)
    indices = torch.randint(0, m, (m,))
    X_bootstrap = X[indices]
    y_bootstrap = y[indices]

    return X_bootstrap, y_bootstrap

def random_forest_fit(X, y, m, k, clf=fit_decision_tree):
    trees = []
    for _ in range(m):
        X_bootstrap, y_bootstrap = bootstrap(X, y)
        tree = clf(X_bootstrap, y_bootstrap)
        trees.append(tree)
    return trees

def random_forest_predict(X, clfs, predict=predict):
    predictions = [predict(tree, X) for tree in clfs]
    predictions_stack = torch.stack(predictions)
    result_mode, _ = torch.mode(predictions_stack, dim=0)

    return result_mode

Test your code again on the `iris` dataset. Our random forest was able to improve the accuracy of the decision tree by about 10\%!

In [178]:
torch.manual_seed(42)
RF = random_forest_fit(X,y,50,2)

print('Train accuracy: ', (random_forest_predict(X, RF) == y).float().mean().item())
print('Test accuracy: ', (random_forest_predict(X_te, RF) == y_te).float().mean().item())

Train accuracy:  1.0
Test accuracy:  0.9333333373069763


As a sanity check, the random forest can get around 95-97% test accuracy.

## 2. Boosting (5pts)

In this problem, you'll implement a basic boosting algorithm on the binary classification breast cancer dataset. Here, we've provided the following weak learner for you: an $\ell_2$ regularized logistic classifier trained with gradient descent.

In [179]:
class Logistic(nn.Module):
    def __init__(self):
        super(Logistic, self).__init__()
        self.linear = nn.Linear(30,1)
    def forward(self, X):
        out = self.linear(X)
        return out.squeeze()

def fit_logistic_clf(X,y):
    clf = Logistic()
    opt = torch.optim.Adam(clf.parameters(), lr=0.1, weight_decay=1e2)
    loss = torch.nn.BCEWithLogitsLoss()
    for t in range(200):
        out = clf(X)
        opt.zero_grad()
        loss(out,(y>0).float()).backward()
        opt.step()
    return clf

def predict_logistic_clf(X, clf):
    return torch.sign(clf(X)).squeeze()

Your task is to boost this logistic classifier to reduce its bias. Implement the following two functions:

+ Finish the boosting algorithm (3pts): we've provided a template for the boosting algorithm in `boosting_fit`, however it is missing several components. Fill in the missing snippets of code.
+ Prediction after boosting (2pts): implement `boosting_predict` to make predictions with a given boosted model.

In [180]:
def boosting_fit(X,y, T=10):
    # X := Tensor(float) of size (m,d) -- Batch of m examples of demension d
    # y := Tensor(int) of size (m) -- the given vectors of labels of the examples
    # T := Maximum number of models to be implemented

    m = X.size(0)
    clfs = []
    mu = torch.full((m,), 1/m, dtype=torch.float)
    while len(clfs) < T:
        # Calculate the weights for each sample mu. You may need to
        # divide this into the base case and the inductive case.
        # mu = ...

        ## ANSWER TODO

        ## END ANSWER

        idx = torch.multinomial(mu, m, replacement=True)
        X0, y0 = X[idx], y[idx]


        clf = fit_logistic_clf(X0, y0)

        # Calculate the epsilon error term
        # eps = ...

        ## ANSWER TODO
        y_pred = predict_logistic_clf(X0, clf)
        errors = y_pred != y0
        eps = torch.sum(mu[errors]) / torch.sum(mu)
        ## END ANSWER

        if eps > 0.5:
            continue

        # Calculate the alpha term here
        # alpha = ...

        ## ANSWER TODO
        alpha = 0.5 * torch.log((1 - eps) / eps)

        mu[errors] *= torch.exp(alpha)
        mu[~errors] *= torch.exp(-alpha)
        mu /= torch.sum(mu)
        ## END ANSWER

        clfs.append((alpha,clf))
    return clfs

def boosting_predict(X, clfs):
    # X := Tensor(float) of size (m,d) -- Batch of m examples of demension d
    # clfs := list of tuples of (float, logistic classifier) -- the list of boosted classifiers
    # Return := Tnesor(int) of size (m) -- the predicted labels of the dataset
    weighted_votes = torch.zeros(X.shape[0], dtype=torch.float32)

    for alpha, clf in clfs:
        predictions = predict_logistic_clf(X, clf)
        weighted_votes += alpha * predictions

    final_labels = (weighted_votes >= 0).int()  # Converts True/False to 1/0
    final_labels = 2 * final_labels - 1
    return final_labels


Test out your code on the breast cancer dataset. As a sanity check, your statndard logistic classifier will get a train/test accuracy of around 84%-88% while the boosted logistic classifier will get a train/test accuracy of around 90%.

In [181]:
from sklearn.datasets import load_breast_cancer
cancer = datasets.load_breast_cancer()
data=train_test_split(cancer.data,cancer.target,test_size=0.2,random_state=123)

torch.manual_seed(123)

X,X_te,y,y_te = [torch.from_numpy(A) for A in data]
X,X_te,y,y_te = X.float(), X_te.float(), torch.sign(y.long()-0.5), torch.sign(y_te.long()-0.5)


logistic_clf = fit_logistic_clf(X,y)
print("Logistic classifier accuracy:")
print('Train accuracy: ', (predict_logistic_clf(X, logistic_clf) == y).float().mean().item())
print('Test accuracy: ', (predict_logistic_clf(X_te, logistic_clf) == y_te).float().mean().item())

boosting_clfs = boosting_fit(X,y)
print("Boosted logistic classifier accuracy:")
print('Train accuracy: ', (boosting_predict(X, boosting_clfs) == y).float().mean().item())
print('Test accuracy: ', (boosting_predict(X_te, boosting_clfs) == y_te).float().mean().item())

Logistic classifier accuracy:
Train accuracy:  0.8351648449897766
Test accuracy:  0.7894737124443054
Boosted logistic classifier accuracy:
Train accuracy:  0.8527472615242004
Test accuracy:  0.9035087823867798


<ipython-input-180-3211c4288a66>:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  predictions = torch.tensor(predictions, dtype=torch.float32)


## Autograder

In [182]:
# Autograder will be announced on Ed Discussion approximately a week after initial release
from penngrader.grader import PennGrader

# PLEASE ENSURE YOUR PENN-ID IS ENTERED CORRECTLY. IF NOT, THE AUTOGRADER WON'T KNOW WHO
# TO ASSIGN POINTS TO YOU IN OUR BACKEND
STUDENT_ID = 76143392 # YOUR PENN-ID GOES HERE AS AN INTEGER #
SECRET = STUDENT_ID

grader = PennGrader('config.yaml', 'cis5200_sp24_HW4', STUDENT_ID, SECRET)


grader.grade(test_case_id = 'entropy', answer = entropy)
grader.grade(test_case_id = 'find_split', answer = find_split)
grader.grade(test_case_id = 'expand_node', answer = expand_node)
grader.grade(test_case_id = 'predict_one', answer = predict_one)

grader.grade(test_case_id = 'bootstrap', answer = bootstrap)
grader.grade(test_case_id = 'random_forest_fit', answer = random_forest_fit)
grader.grade(test_case_id = 'random_forest_predict', answer = random_forest_predict)

grader.grade(test_case_id = 'boosting_fit', answer = boosting_fit)
grader.grade(test_case_id = 'boosting_predict', answer = boosting_predict)

PennGrader initialized with Student ID: 76143392

Make sure this correct or we will not be able to store your grade
Correct! You earned 2/2 points. You are a star!

Your submission has been successfully recorded in the gradebook.
Correct! You earned 1/1 points. You are a star!

Your submission has been successfully recorded in the gradebook.
Correct! You earned 2/2 points. You are a star!

Your submission has been successfully recorded in the gradebook.
Correct! You earned 2/2 points. You are a star!

Your submission has been successfully recorded in the gradebook.
Correct! You earned 1/1 points. You are a star!

Your submission has been successfully recorded in the gradebook.
Correct! You earned 1/1 points. You are a star!

Your submission has been successfully recorded in the gradebook.
Correct! You earned 1/1 points. You are a star!

Your submission has been successfully recorded in the gradebook.
Correct! You earned 3/3 points. You are a star!

Your submission has been successfully

# Submitting to Gradescope
Before submitting to Gradescope, make sure that selecting "Runtime" -> "Restart and run all" completes all cells without errors.

1. Go to the File menu and choose "Download .ipynb" and also "Download .py". Make sure these files are named homework4.ipynb and homework4.py, respectively
2. Go to GradeScope through the canvas page and ensure your class is "BAN_CIS-5200-001 202310"
3. Select Homework 4
4. Upload both files (the .ipynb and the .py)
5. PLEASE CHECK THE AUTOGRADER OUTPUT TO ENSURE YOUR SUBMISSION IS PROCESSED CORRECTLY! If this is the case, you should be all set with the programming component of this homework!